# 📊 Payroll Reconciliation Tool
**GL Report vs Payroll Register — Automated Comparison**

---
### Workflow
| Step | Action |
|------|--------|
| 1 | Define reconciliation configuration (GL codes ↔ Pay codes) |
| 2 | Upload GL Report → map columns |
| 3 | Upload Payroll Register → map columns |
| 4 | Run reconciliation |
| 5 | Download 6-sheet Excel report |

> **Run each cell in order top-to-bottom.**

In [ ]:
# ── Cell 1: Install & Imports ─────────────────────────────────────────────────
!pip install -q xlsxwriter openpyxl pyxlsb

import io, re, datetime
import pandas as pd
import numpy as np
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import files

# ── Global state ──────────────────────────────────────────────────────────────
state = {
    'gl_df'      : None,
    'pr_df'      : None,
    'gl_cols'    : {},
    'pr_cols'    : {},
    'config_rows': [],
    'results'    : None,
}

# ── Helper: smart header detection ───────────────────────────────────────────
def detect_header_row(raw_df, max_scan=15):
    KEYWORDS = ['code','type','date','amount','amt','name','id','source','title',
                'pay','gl','steps','recon','tax','earn','bene','deduc','net','empl']
    n_cols = len(raw_df.columns)
    best_row, best_score = 0, -1.0
    for i in range(min(max_scan, len(raw_df))):
        row  = raw_df.iloc[i]
        vals = [str(v).strip() for v in row
                if pd.notna(v) and str(v).strip() not in ('', 'nan')]
        if not vals or len(vals) / max(n_cols, 1) < 0.3:
            continue
        score = 0.0
        for v in vals:
            if   len(v) <= 50: score += 2.0
            elif len(v) <= 80: score += 0.5
            else:              score -= 1.0
            try:    float(v.replace(',', '')); score -= 2.0
            except: score += 0.5
            if re.match(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', v): score -= 1.5
            if re.search(r'[A-Z][a-z]|[a-z][A-Z]', v):         score += 1.5
            if '_' in v:                                         score += 1.0
            if v.isupper() and len(v) > 1:                      score += 1.0
            if any(kw in v.lower() for kw in KEYWORDS):         score += 2.0
        total = (len(vals) / max(n_cols, 1)) * (score / len(vals))
        if total > best_score:
            best_score = total
            best_row   = i
    return best_row

# ── Helper: read raw top rows (no header parsing) ────────────────────────────
def _read_raw(content, filename, nrows=15):
    fname = filename.lower()
    kw = dict(header=None, dtype=str, nrows=nrows)
    eng = 'odf' if fname.endswith('.ods') else ('pyxlsb' if fname.endswith('.xlsb') else None)
    if fname.endswith('.csv'):
        return pd.read_csv(io.BytesIO(content), encoding_errors='replace',
                           on_bad_lines='skip', **kw)
    if fname.endswith(('.tsv', '.txt')):
        return pd.read_csv(io.BytesIO(content), sep='\t',
                           encoding_errors='replace', on_bad_lines='skip', **kw)
    return pd.read_excel(io.BytesIO(content), **(dict(engine=eng) if eng else {}), **kw)

# ── Helper: read file with a forced header row (0-indexed) ───────────────────
def read_file_forced(content, filename, header_row):
    """Read file treating `header_row` (0-indexed) as the column header row."""
    fname = filename.lower()
    try:
        kw = dict(header=header_row, dtype=str)
        eng = 'odf' if fname.endswith('.ods') else ('pyxlsb' if fname.endswith('.xlsb') else None)
        if fname.endswith('.csv'):
            df = pd.read_csv(io.BytesIO(content), encoding_errors='replace',
                             on_bad_lines='skip', **kw)
        elif fname.endswith(('.tsv', '.txt')):
            df = pd.read_csv(io.BytesIO(content), sep='\t',
                             encoding_errors='replace', on_bad_lines='skip', **kw)
        else:
            df = pd.read_excel(io.BytesIO(content),
                               **(dict(engine=eng) if eng else {}), **kw)
    except Exception as e:
        print(f'Error reading file: {e}')
        return None
    # Clean columns
    new_cols, seen = [], {}
    for col in df.columns:
        c = str(col).strip()
        if c.lower().startswith('unnamed:') or c in ('nan', ''): continue
        if c in seen: seen[c] += 1; c = f'{c}_{seen[c]}'
        else: seen[c] = 0
        new_cols.append((col, c))
    if not new_cols:
        return None
    df = df[[orig for orig, _ in new_cols]].copy()
    df.columns = [clean for _, clean in new_cols]
    df.dropna(how='all', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

# ── Helper: read file (auto-detect header) ────────────────────────────────────
def read_file_smart(content, filename):
    try:
        raw = _read_raw(content, filename)
        hdr = detect_header_row(raw)
        df  = read_file_forced(content, filename, hdr)
    except Exception as e:
        print(f'Error reading file: {e}'); return None, 0
    return df, hdr

print('✅ Libraries loaded. Proceed to Step 1.')

---
## Step 1 — Reconciliation Configuration
Edit `CONFIG_ROWS` below to match your client's GL accounts and pay codes, then run the cell.

| Field | Description |
|-------|-------------|
| `recon_step` | Reconciliation step label (e.g. `A. Earnings / Gross Wages`) |
| `gl_code` | GL account code (e.g. `5000`) |
| `gl_title` | GL account description |
| `pay_code` | Payroll pay code (e.g. `Wages`, `MC`, `FIT`) |
| `pay_code_title` | Human-readable label |
| `amount_column` | `EarnAmt` \| `BeneAmt` \| `DeducAmt` \| `EETax` \| `ERTax` \| `EETax & ERTax` \| `NetAmt` |
| `code_type` | `EARNING` \| `BENEFIT` \| `DEDUCT` \| `TAXES` \| *(leave empty for bank)* |

In [ ]:
# ── Cell 2: Reconciliation Configuration ─────────────────────────────────────
# ✏️  EDIT THESE ROWS to match your client. Add / remove / change as needed.

CLIENT_NAME  = 'default'      # e.g. 'Acme Corp'
PERIOD_LABEL = ''             # e.g. 'January 2026'

CONFIG_ROWS = [
    # ── A. Earnings ─────────────────────────────────────────────────────────
    dict(recon_step='A. Earnings / Gross Wages',      gl_code='5000', gl_title='Salaries & Wages',
         pay_code='Wages',    pay_code_title='Regular Wages',          amount_column='EarnAmt',       code_type='EARNING'),
    dict(recon_step='A. Earnings / Gross Wages',      gl_code='5001', gl_title='Overtime Wages',
         pay_code='OT',       pay_code_title='Overtime',               amount_column='EarnAmt',       code_type='EARNING'),

    # ── B. Benefits — ER Expense ─────────────────────────────────────────────
    dict(recon_step='B. Benefits / ER Expense',       gl_code='5130', gl_title='Health Insurance Expense',
         pay_code='Health16', pay_code_title='Health Insurance ER',    amount_column='BeneAmt',       code_type='BENEFIT'),
    dict(recon_step='B. Benefits / ER Expense',       gl_code='5140', gl_title='Retirement Expense',
         pay_code='401K',     pay_code_title='401K Employer Match',    amount_column='BeneAmt',       code_type='BENEFIT'),

    # ── B.1 Benefit Liabilities ──────────────────────────────────────────────
    dict(recon_step='B.1 Benefit Liabilities',        gl_code='2145', gl_title='Health Insurance Payable',
         pay_code='Health16', pay_code_title='Health Insurance',       amount_column='BeneAmt',       code_type='BENEFIT'),
    dict(recon_step='B.1 Benefit Liabilities',        gl_code='2148', gl_title='Retirement Payable',
         pay_code='401K',     pay_code_title='401K',                   amount_column='BeneAmt',       code_type='BENEFIT'),

    # ── C. Deductions / EE Liabilities ──────────────────────────────────────
    dict(recon_step='C. Deductions / EE Liabilities', gl_code='2141', gl_title='Health Insurance EE',
         pay_code='Health16', pay_code_title='Health 2017 Ins Deduction', amount_column='DeducAmt',  code_type='DEDUCT'),
    dict(recon_step='C. Deductions / EE Liabilities', gl_code='2142', gl_title='Dental Insurance EE',
         pay_code='Dental16', pay_code_title='Dental 2016 Ins Deduction', amount_column='DeducAmt',  code_type='DEDUCT'),
    dict(recon_step='C. Deductions / EE Liabilities', gl_code='2143', gl_title='Vision Insurance EE',
         pay_code='Vision16', pay_code_title='Vision 2016',               amount_column='DeducAmt',  code_type='DEDUCT'),

    # ── D. EE & ER Tax Liabilities ──────────────────────────────────────────
    dict(recon_step='D. EE & ER Tax Liabilities',     gl_code='2115', gl_title='Federal Payroll Taxes Payable',
         pay_code='FIT',  pay_code_title='Federal Withholding',         amount_column='EETax',         code_type='TAXES'),
    dict(recon_step='D. EE & ER Tax Liabilities',     gl_code='2115', gl_title='Federal Payroll Taxes Payable',
         pay_code='MC',   pay_code_title='Medicare',                   amount_column='EETax & ERTax', code_type='TAXES'),
    dict(recon_step='D. EE & ER Tax Liabilities',     gl_code='2115', gl_title='Federal Payroll Taxes Payable',
         pay_code='SS',   pay_code_title='Social Security',            amount_column='EETax & ERTax', code_type='TAXES'),
    dict(recon_step='D. EE & ER Tax Liabilities',     gl_code='2120', gl_title='State Payroll Taxes Payable',
         pay_code='SWT',  pay_code_title='State Withholding',          amount_column='EETax',         code_type='TAXES'),

    # ── E. ER Tax Expense (FICA) ────────────────────────────────────────────
    dict(recon_step='E. ER Tax Expense (FICA)',        gl_code='5100', gl_title='FICA Expense',
         pay_code='MC',   pay_code_title='Medicare ER',               amount_column='ERTax',         code_type='TAXES'),
    dict(recon_step='E. ER Tax Expense (FICA)',        gl_code='5100', gl_title='FICA Expense',
         pay_code='SS',   pay_code_title='Social Security ER',        amount_column='ERTax',         code_type='TAXES'),

    # ── F. Bank Payment ──────────────────────────────────────────────────────
    dict(recon_step='F. Bank Payment to Employee',    gl_code='1020', gl_title='Cash in Bank - Payroll',
         pay_code='',     pay_code_title='Net Pay',                   amount_column='NetAmt',        code_type=''),
]

state['config_rows'] = CONFIG_ROWS
cfg_df = pd.DataFrame(CONFIG_ROWS)

print(f'✅ Configuration loaded — {len(CONFIG_ROWS)} rows  |  Client: {CLIENT_NAME}  |  Period: {PERIOD_LABEL or "(not set)"}')
print()

display(cfg_df.style
    .set_caption('Reconciliation Configuration (edit CONFIG_ROWS above and re-run)')
    .set_properties(**{'font-size': '11px'})
    .map(lambda v: 'background-color:#E8F5E9;font-weight:bold', subset=['gl_code', 'pay_code'])
    .map(lambda v: 'background-color:#FFF3E0;font-weight:bold', subset=['code_type'])
    .map(lambda v: 'background-color:#EEF4FF', subset=['amount_column'])
)

---
## Step 2 — Upload GL Report
Supported formats: `.xlsx` `.xls` `.xlsm` `.xlsb` `.csv` `.tsv` `.txt` `.ods`

In [ ]:
# ── Cell 3: Upload GL Report ─────────────────────────────────────────────────
print('📂 Select your GL Report file ...')
uploaded = files.upload()

if not uploaded:
    print('No file selected.')
else:
    fname     = list(uploaded.keys())[0]
    raw_bytes = uploaded[fname]

    # ── Read raw top rows so user can verify the header position ─────────────
    try:
        _raw = _read_raw(raw_bytes, fname, nrows=15)
        hdr  = detect_header_row(_raw)
    except Exception as e:
        print(f'❌ Cannot read file: {e}'); _raw = None; hdr = 0

    if _raw is not None:
        print(f'\n✅ {fname}')
        print(f'   Auto-detected header at row {hdr} (0-indexed = row {hdr + 1} in the file)')
        print('\n📋 Raw file preview — find the row that contains your column names:')

        # Show raw rows with detected header highlighted
        _pv = _raw.fillna('').copy()
        _pv.index = [
            f'Row {i}  ◀ HEADER DETECTED' if i == hdr else f'Row {i}'
            for i in range(len(_pv))
        ]
        display(_pv.style.apply(
            lambda s: [
                'background-color:#BDD7EE;font-weight:bold;color:#1F3864'
                if '◀ HEADER' in str(s.name) else ''
                for _ in s
            ], axis=1
        ))

        # ── Header row override ───────────────────────────────────────────────
        hdr_input = widgets.BoundedIntText(
            value=hdr, min=0, max=max(14, hdr + 5),
            description='Header row (0-indexed):',
            style={'description_width': '185px'},
            layout=widgets.Layout(width='360px')
        )
        btn_apply = widgets.Button(
            description='Apply & Show Column Mapping →',
            button_style='primary',
            layout=widgets.Layout(width='270px', margin='6px 0 2px')
        )
        hint_lbl = widgets.HTML(
            '<span style="font-size:11px;color:#555">'
            'Row 0 = first row of the file.  '
            'Change the number above if the detected row is wrong, then click Apply.'
            '</span>'
        )
        out_mapping = widgets.Output()

        def apply_gl_header(btn):
            chosen = hdr_input.value
            df = read_file_forced(raw_bytes, fname, chosen)
            with out_mapping:
                clear_output()
                if df is None:
                    print(f'❌ Could not read with header row {chosen}. Try a different row.')
                    return
                state['gl_df'] = df
                cols = list(df.columns)
                print(f'✅ Loaded with header row {chosen}  |  Rows: {len(df):,}  |  Columns: {len(cols)}')
                print('\n📋 Preview (first 5 rows):')
                display(df.head(5))

                # ── Column role dropdowns ──────────────────────────────────────
                NONE = '— skip —'
                GL_ROLES = [
                    ('trans_source',  'Transaction Source ℹ',  'Filters PRS payroll rows  (e.g. TransSource)',   ['source','trans','tran','src']),
                    ('gl_code',       'GL Code / Account  ✱',  'Account code  (e.g. AcctCode, GLCode)',          ['code','acct','account','gl_c']),
                    ('gl_title',      'GL Title / Name    ✱',  'Account description  (e.g. AcctTitle)',          ['title','name','desc','acct']),
                    ('net_amount',    'Net Amount         ✱',  'Net / balance / debit amount column',            ['net','amount','amt','balance','debit']),
                    ('debit_amount',  'Debit Amount',           'Debit column if separate from Net  (optional)',  ['debit','dr']),
                    ('credit_amount', 'Credit Amount',          'Credit column if separate from Net  (optional)', ['credit','cr']),
                ]

                dropdowns_gl = {}
                widget_rows  = []
                for role, label, hint, kws in GL_ROLES:
                    dd = widgets.Dropdown(options=[NONE] + cols,
                                         layout=widgets.Layout(width='300px'))
                    for kw in kws:
                        m = [c for c in cols if kw in c.lower()]
                        if m: dd.value = m[0]; break
                    dropdowns_gl[role] = dd
                    lbl = widgets.HTML(
                        f'<div style="width:240px">'
                        f'<b style="font-size:12px">{label}</b><br>'
                        f'<span style="font-size:10px;color:#777">{hint}</span></div>'
                    )
                    widget_rows.append(widgets.HBox([lbl, dd]))

                out_confirm = widgets.Output()

                def confirm_gl(btn):
                    mapping = {role: dd.value for role, dd in dropdowns_gl.items()
                               if dd.value != NONE}
                    required = {'gl_code', 'gl_title', 'net_amount'}
                    missing  = required - set(mapping.keys())
                    with out_confirm:
                        clear_output()
                        if missing:
                            print(f'❌ Please assign required roles: {", ".join(missing)}')
                        else:
                            state['gl_cols'] = mapping
                            print('✅ GL column mapping confirmed:')
                            for r, c in mapping.items():
                                print(f'   {r:20s} → {c}')

                btn_gl = widgets.Button(
                    description='✅ Confirm GL Columns',
                    button_style='success',
                    layout=widgets.Layout(width='220px', margin='14px 0 4px')
                )
                btn_gl.on_click(confirm_gl)

                print('\n🔧 Map GL Report columns to required roles  (✱ = required):')
                display(widgets.VBox(widget_rows + [btn_gl, out_confirm]))

        btn_apply.on_click(apply_gl_header)

        display(widgets.VBox([hdr_input, hint_lbl, btn_apply, out_mapping]))
        apply_gl_header(None)   # auto-apply the detected header

---
## Step 3 — Upload Payroll Register
Supported formats: `.xlsx` `.xls` `.xlsm` `.xlsb` `.csv` `.tsv` `.txt` `.ods`

In [ ]:
# ── Cell 4: Upload Payroll Register ──────────────────────────────────────────
print('📂 Select your Payroll Register file ...')
uploaded = files.upload()

if not uploaded:
    print('No file selected.')
else:
    fname     = list(uploaded.keys())[0]
    raw_bytes = uploaded[fname]

    # ── Read raw top rows so user can verify the header position ─────────────
    try:
        _raw = _read_raw(raw_bytes, fname, nrows=15)
        hdr  = detect_header_row(_raw)
    except Exception as e:
        print(f'❌ Cannot read file: {e}'); _raw = None; hdr = 0

    if _raw is not None:
        print(f'\n✅ {fname}')
        print(f'   Auto-detected header at row {hdr} (0-indexed = row {hdr + 1} in the file)')
        print('\n📋 Raw file preview — find the row that contains your column names:')

        _pv = _raw.fillna('').copy()
        _pv.index = [
            f'Row {i}  ◀ HEADER DETECTED' if i == hdr else f'Row {i}'
            for i in range(len(_pv))
        ]
        display(_pv.style.apply(
            lambda s: [
                'background-color:#BDD7EE;font-weight:bold;color:#1F3864'
                if '◀ HEADER' in str(s.name) else ''
                for _ in s
            ], axis=1
        ))

        # ── Header row override ───────────────────────────────────────────────
        hdr_input = widgets.BoundedIntText(
            value=hdr, min=0, max=max(14, hdr + 5),
            description='Header row (0-indexed):',
            style={'description_width': '185px'},
            layout=widgets.Layout(width='360px')
        )
        btn_apply = widgets.Button(
            description='Apply & Show Column Mapping →',
            button_style='primary',
            layout=widgets.Layout(width='270px', margin='6px 0 2px')
        )
        hint_lbl = widgets.HTML(
            '<span style="font-size:11px;color:#555">'
            'Row 0 = first row of the file.  '
            'Change the number above if the detected row is wrong, then click Apply.'
            '</span>'
        )
        out_mapping = widgets.Output()

        def apply_pr_header(btn):
            chosen = hdr_input.value
            df = read_file_forced(raw_bytes, fname, chosen)
            with out_mapping:
                clear_output()
                if df is None:
                    print(f'❌ Could not read with header row {chosen}. Try a different row.')
                    return
                state['pr_df'] = df
                cols = list(df.columns)
                print(f'✅ Loaded with header row {chosen}  |  Rows: {len(df):,}  |  Columns: {len(cols)}')
                print('\n📋 Preview (first 5 rows):')
                display(df.head(5))

                # ── Column role dropdowns ──────────────────────────────────────
                NONE = '— skip —'
                PR_ROLES = [
                    ('pay_code',         'Pay Code           ✱', 'Pay code column  (e.g. PayCode, Code)',          ['code','pay','paycode']),
                    ('code_type',        'Code Type          ✱', 'EARNING / BENEFIT / DEDUCT / TAXES column',      ['type','codetype']),
                    ('earn_amount',      'Earnings Amount    ✱', 'EarnAmt — gross wages column',                   ['earn','earnamt']),
                    ('benefit_amount',   'Benefit Amount     ✱', 'BeneAmt — employer benefit cost',                ['bene','beneamt']),
                    ('deduction_amount', 'Deduction Amount   ✱', 'DeducAmt — employee deductions',                 ['deduc','deducamt']),
                    ('ee_tax',           'Employee Tax       ✱', 'EETax — employee tax withholding',               ['eetax','ee_tax','eetx']),
                    ('er_tax',           'Employer Tax       ✱', 'ERTax — employer tax contributions',             ['ertax','er_tax','ertx']),
                    ('net_amount',       'Net Pay Amount',        'NetAmt — for bank cross-check  (optional)',      ['net','netamt']),
                ]

                dropdowns_pr = {}
                widget_rows  = []
                for role, label, hint, kws in PR_ROLES:
                    dd = widgets.Dropdown(options=[NONE] + cols,
                                         layout=widgets.Layout(width='300px'))
                    for kw in kws:
                        m = [c for c in cols if kw in c.lower()]
                        if m: dd.value = m[0]; break
                    dropdowns_pr[role] = dd
                    lbl = widgets.HTML(
                        f'<div style="width:240px">'
                        f'<b style="font-size:12px">{label}</b><br>'
                        f'<span style="font-size:10px;color:#777">{hint}</span></div>'
                    )
                    widget_rows.append(widgets.HBox([lbl, dd]))

                out_confirm = widgets.Output()

                def confirm_pr(btn):
                    mapping = {role: dd.value for role, dd in dropdowns_pr.items()
                               if dd.value != NONE}
                    required = {'pay_code', 'code_type', 'earn_amount', 'benefit_amount',
                                'deduction_amount', 'ee_tax', 'er_tax'}
                    missing  = required - set(mapping.keys())
                    with out_confirm:
                        clear_output()
                        if missing:
                            print(f'❌ Please assign required roles: {", ".join(missing)}')
                        else:
                            state['pr_cols'] = mapping
                            print('✅ Payroll Register column mapping confirmed:')
                            for r, c in mapping.items():
                                print(f'   {r:22s} → {c}')

                btn_pr = widgets.Button(
                    description='✅ Confirm PR Columns',
                    button_style='success',
                    layout=widgets.Layout(width='230px', margin='14px 0 4px')
                )
                btn_pr.on_click(confirm_pr)

                print('\n🔧 Map Payroll Register columns to required roles  (✱ = required):')
                display(widgets.VBox(widget_rows + [btn_pr, out_confirm]))

        btn_apply.on_click(apply_pr_header)

        display(widgets.VBox([hdr_input, hint_lbl, btn_apply, out_mapping]))
        apply_pr_header(None)   # auto-apply the detected header

---
## Step 3b — Discover Codes (run after Steps 2 & 3)
Run this cell to see the actual GL account codes and Payroll pay codes in your uploaded files.
Copy these values into `CONFIG_ROWS` in Step 1.

In [ ]:
# ── Step 3b: Discover Codes — run after uploading both files ─────────────────
# Shows every unique GL code + title, and every unique pay code + code type.
# Copy the real codes from here into CONFIG_ROWS in Cell 2, then re-run from Cell 2.

assert state['gl_df'] is not None and state['gl_cols'].get('gl_code'), \
    '❌ GL file not uploaded / column not confirmed — run Step 2 first'
assert state['pr_df'] is not None and state['pr_cols'].get('pay_code'), \
    '❌ PR file not uploaded / column not confirmed — run Step 3 first'

gl_df   = state['gl_df']
pr_df   = state['pr_df']
gl_cols = state['gl_cols']
pr_cols = state['pr_cols']

# ── GL: unique account codes + titles ─────────────────────────────────────────
gc_col  = gl_cols['gl_code']
gt_col  = gl_cols.get('gl_title', '')

gl_codes = (
    gl_df[[gc_col, gt_col]].drop_duplicates()
    .rename(columns={gc_col: 'GL Code', gt_col: 'GL Title'})
    .sort_values('GL Code')
    .reset_index(drop=True)
) if gt_col and gt_col in gl_df.columns else (
    gl_df[[gc_col]].drop_duplicates()
    .rename(columns={gc_col: 'GL Code'})
    .assign(**{'GL Title': ''})
    .sort_values('GL Code')
    .reset_index(drop=True)
)

# ── GL columns overview — sample values to help identify roles ────────────────
print('─' * 60)
print('  GL FILE — column overview (first 3 unique values each)')
print('─' * 60)
for col in gl_df.columns:
    samples = gl_df[col].dropna().astype(str).str.strip()
    samples = [v for v in samples.unique() if v not in ('', 'nan')][:3]
    print(f'  {col:<30s}  {samples}')

print()
print('─' * 60)
print(f'  GL UNIQUE ACCOUNT CODES  ({len(gl_codes)} found)')
print('─' * 60)
display(gl_codes.style
    .set_properties(**{'font-size': '11px'})
    .map(lambda v: 'background-color:#E8F5E9;font-weight:bold', subset=['GL Code']))

# ── PR: unique pay codes + code types ─────────────────────────────────────────
pc_col  = pr_cols['pay_code']
ct_col  = pr_cols.get('code_type', '')

pr_codes = (
    pr_df[[pc_col, ct_col]].drop_duplicates()
    .rename(columns={pc_col: 'Pay Code', ct_col: 'Code Type'})
    .sort_values(['Code Type', 'Pay Code'])
    .reset_index(drop=True)
) if ct_col and ct_col in pr_df.columns else (
    pr_df[[pc_col]].drop_duplicates()
    .rename(columns={pc_col: 'Pay Code'})
    .assign(**{'Code Type': ''})
    .sort_values('Pay Code')
    .reset_index(drop=True)
)

print()
print('─' * 60)
print('  PR FILE — column overview (first 3 unique values each)')
print('─' * 60)
for col in pr_df.columns:
    samples = pr_df[col].dropna().astype(str).str.strip()
    samples = [v for v in samples.unique() if v not in ('', 'nan')][:3]
    print(f'  {col:<30s}  {samples}')

print()
print('─' * 60)
print(f'  PR UNIQUE PAY CODES  ({len(pr_codes)} found)')
print('─' * 60)
display(pr_codes.style
    .set_properties(**{'font-size': '11px'})
    .map(lambda v: 'background-color:#E8F5E9;font-weight:bold', subset=['Pay Code'])
    .map(lambda v: 'background-color:#FFF3E0;font-weight:bold', subset=['Code Type']))

print()
print('=' * 60)
print('  ACTION: Go back to Cell 2 (Step 1) and update CONFIG_ROWS')
print('  using the GL codes and Pay codes shown above.')
print('  Then re-run Cell 2 and Cell 5 (Step 4).')
print('=' * 60)

---
## Step 4 — Run Reconciliation
Run this cell after confirming both file column mappings above.

In [ ]:
# ── Cell 5: Run Reconciliation ────────────────────────────────────────────────
assert state['gl_df']  is not None,         '❌ GL Report not uploaded'
assert state['gl_cols'].get('gl_code'),      '❌ GL columns not confirmed — run Step 2 cell'
assert state['pr_df']  is not None,         '❌ Payroll Register not uploaded'
assert state['pr_cols'].get('pay_code'),     '❌ PR columns not confirmed — run Step 3 cell'
assert state['config_rows'],                '❌ Config is empty — run Step 1 cell'

gl_df    = state['gl_df'].copy()
pr_df    = state['pr_df'].copy()
gl_cols  = state['gl_cols']
pr_cols  = state['pr_cols']
cfg_rows = state['config_rows']
TOLERANCE = 0.01

def to_num(series):
    return pd.to_numeric(series.astype(str).str.replace(',', ''), errors='coerce').fillna(0)

SEP = '─' * 65

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — GL: filter PRS rows, parse amounts, map to recon steps
# ══════════════════════════════════════════════════════════════════════════════
print(SEP)
print('  STEP 1 — GL Processing')
print(SEP)

total_gl_rows = len(gl_df)
if gl_cols.get('trans_source'):
    mask  = gl_df[gl_cols['trans_source']].astype(str).str.contains('PRS', case=False, na=False)
    gl_df = gl_df[mask].copy()
    print(f'  Source filter (PRS): {total_gl_rows:,} total rows → {len(gl_df):,} PRS rows kept')
else:
    print(f'  No Transaction Source column mapped — using all {total_gl_rows:,} rows')

gl_df['_net']      = to_num(gl_df[gl_cols['net_amount']])
gl_df['_gl_code']  = gl_df[gl_cols['gl_code']].astype(str).str.strip()
gl_df['_gl_title'] = gl_df[gl_cols['gl_title']].astype(str).str.strip()

# Build GL lookup from CONFIG_ROWS
gl_lookup = {}
for row in cfg_rows:
    gc = str(row['gl_code']).strip()
    if gc and gc not in gl_lookup:
        gl_lookup[gc] = {
            'gl_title'     : row['gl_title'],
            'recon_step'   : row['recon_step'],
            'amount_column': row['amount_column'],
        }

config_gl_codes  = set(gl_lookup.keys())
actual_gl_codes  = set(gl_df['_gl_code'].unique())
matched_gl_codes = config_gl_codes & actual_gl_codes
missing_in_file  = config_gl_codes - actual_gl_codes
unmapped_in_file = actual_gl_codes - config_gl_codes

print(f'\n  CONFIG_ROWS GL codes   : {sorted(config_gl_codes)}')
print(f'  GL file unique codes   : {sorted(actual_gl_codes)}')
print(f'  ✓ Matched GL codes     : {sorted(matched_gl_codes)}')
if missing_in_file:
    print(f'  ⚠ In config but NOT in file : {sorted(missing_in_file)}')
if unmapped_in_file:
    print(f'  ⚠ In file but NOT in config : {sorted(unmapped_in_file)}')
    print(f'    → These rows will show as "Unmapped" in the output.')
    print(f'      Add them to CONFIG_ROWS in Cell 2 if they should be reconciled.')

gl_df['_recon'] = gl_df['_gl_code'].map(
    lambda c: gl_lookup.get(c, {}).get('recon_step', 'Unmapped'))

n_mapped_gl   = (gl_df['_recon'] != 'Unmapped').sum()
n_unmapped_gl = (gl_df['_recon'] == 'Unmapped').sum()
print(f'\n  GL rows mapped   : {n_mapped_gl:,}')
print(f'  GL rows unmapped : {n_unmapped_gl:,}')

gl_pivot = (
    gl_df.groupby(['_gl_code', '_gl_title', '_recon'])['_net']
    .sum().reset_index()
    .rename(columns={
        '_gl_code' : 'GL Code',
        '_gl_title': 'GL Title',
        '_recon'   : 'Reconciliation Mapping',
        '_net'     : 'Sum of Net Amount',
    })
)

gl_mapped = (gl_df.drop(columns=['_net','_gl_code','_gl_title'], errors='ignore')
                  .rename(columns={'_recon': 'Reconciliation Mapping'}))

print(f'\n  GL Pivot ({len(gl_pivot)} rows):')
display(gl_pivot)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Payroll Register: parse amounts, map pay codes → GL codes
# ══════════════════════════════════════════════════════════════════════════════
print()
print(SEP)
print('  STEP 2 — Payroll Register Processing')
print(SEP)

AMT_ROLES = {
    'earn_amount'     : 'Sum EarnAmt',
    'benefit_amount'  : 'Sum BeneAmt',
    'deduction_amount': 'Sum DeducAmt',
    'ee_tax'          : 'Sum EETax',
    'er_tax'          : 'Sum ERTax',
    'net_amount'      : 'Sum NetAmt',
}
for role, pivot_col in AMT_ROLES.items():
    col = pr_cols.get(role)
    pr_df[pivot_col] = to_num(pr_df[col]) if col and col in pr_df.columns else 0.0
    status = f'→ {col}' if col and col in pr_df.columns else '(not mapped)'
    print(f'  {pivot_col:<18s} {status}')

# PR lookup: (pay_code, code_type) → list of GL codes
pr_to_gl = {}
for row in cfg_rows:
    key = (str(row['pay_code']).strip(), str(row['code_type']).strip())
    pr_to_gl.setdefault(key, []).append(str(row['gl_code']).strip())

pr_df['_pay_code']  = pr_df[pr_cols['pay_code']].astype(str).str.strip()
pr_df['_code_type'] = pr_df[pr_cols['code_type']].astype(str).str.strip() \
                      if pr_cols.get('code_type') else ''

actual_pr_keys   = set(zip(pr_df['_pay_code'], pr_df['_code_type']))
config_pr_keys   = set(pr_to_gl.keys())
matched_pr_keys  = config_pr_keys & actual_pr_keys
unmapped_pr_keys = actual_pr_keys - config_pr_keys

print(f'\n  PR file (pay_code, code_type) combinations: {len(actual_pr_keys)}')
print(f'  CONFIG_ROWS combinations                  : {len(config_pr_keys)}')
print(f'  ✓ Matched combinations                    : {len(matched_pr_keys)}')
if unmapped_pr_keys:
    print(f'  ⚠ In PR but NOT in config ({len(unmapped_pr_keys)} combos):')
    for k in sorted(unmapped_pr_keys)[:20]:
        print(f'      pay_code={k[0]!r:20s}  code_type={k[1]!r}')
    if len(unmapped_pr_keys) > 20:
        print(f'      ... and {len(unmapped_pr_keys)-20} more')

def pr_recon_label(row):
    gls = pr_to_gl.get((row['_pay_code'], row['_code_type']), [])
    return ' | '.join(gls) if gls else 'Unmapped'

pr_df['Reconciliation Mapping'] = pr_df.apply(pr_recon_label, axis=1)

n_mapped_pr   = (pr_df['Reconciliation Mapping'] != 'Unmapped').sum()
n_unmapped_pr = (pr_df['Reconciliation Mapping'] == 'Unmapped').sum()
print(f'\n  PR rows mapped   : {n_mapped_pr:,}')
print(f'  PR rows unmapped : {n_unmapped_pr:,}')

pr_pivot = (pr_df
    .groupby('Reconciliation Mapping')[list(AMT_ROLES.values())]
    .sum().reset_index())

pr_mapped    = pr_df.drop(columns=['_pay_code','_code_type'], errors='ignore')
pr_net_total = float(pr_df['Sum NetAmt'].sum())
print(f'\n  Net Pay total (bank cross-check): ${pr_net_total:,.2f}')

print(f'\n  PR Pivot ({len(pr_pivot)} rows):')
display(pr_pivot)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Reconciliation: GL vs PR comparison
# ══════════════════════════════════════════════════════════════════════════════
print()
print(SEP)
print('  STEP 3 — Reconciliation Comparison (GL vs PR)')
print(SEP)

AMT_COL_MAP = {
    'earnamt'      : 'Sum EarnAmt',
    'beneamt'      : 'Sum BeneAmt',
    'deducamt'     : 'Sum DeducAmt',
    'eetax'        : 'Sum EETax',
    'ertax'        : 'Sum ERTax',
    'eetax & ertax': 'BOTH',
    'netamt'       : 'Sum NetAmt',
}
LIAB_KW = ['liabilit','payable','deduction','ee retir','b.1','b1','d. emp']

rows_out = []
for _, gl_row in gl_pivot.iterrows():
    gc     = str(gl_row['GL Code']).strip()
    title  = str(gl_row['GL Title']).strip()
    recon  = str(gl_row['Reconciliation Mapping']).strip()
    gl_net = float(gl_row['Sum of Net Amount'])
    meta   = gl_lookup.get(gc, {})
    pr_col = AMT_COL_MAP.get(meta.get('amount_column','').strip().lower(), '')

    print(f'\n  GL {gc} | {title[:35]:<35s} | recon={recon[:30]:<30s} | GL=${gl_net:>14,.2f} | pr_col={pr_col}')

    # Bank cross-check
    if pr_col == 'Sum NetAmt' or 'bank' in recon.lower():
        variance = round(gl_net + pr_net_total, 2)
        status   = '✓ Match' if abs(variance) < TOLERANCE else '⚠ Variance'
        print(f'         → Bank cross-check: GL {gl_net:,.2f} + Net {pr_net_total:,.2f} = Variance {variance:,.2f}  [{status}]')
        rows_out.append({'Reconciliation Step': recon, 'GL Code': gc, 'GL Title': title,
                         'GL Net Amount': round(gl_net,2), 'PR Amount': round(-pr_net_total,2),
                         'Variance': variance, 'Status': status, 'Notes': 'Bank / Net Pay cross-check'})
        continue

    # Match PR pivot rows by GL code
    mask     = pr_pivot['Reconciliation Mapping'].str.contains(
                   rf'(?<!\d){re.escape(gc)}(?!\d)', regex=True, na=False)
    match_pr = pr_pivot[mask]

    if match_pr.empty:
        print(f'         → No PR mapping found for GL code {gc!r} — check CONFIG_ROWS pay_code/code_type')
        rows_out.append({'Reconciliation Step': recon, 'GL Code': gc, 'GL Title': title,
                         'GL Net Amount': round(gl_net,2), 'PR Amount': 0.0,
                         'Variance': round(gl_net,2), 'Status': '⚠ No PR Match',
                         'Notes': 'GL code not found in PR mapping'})
        continue

    pr_amount = 0.0
    for _, pr_row in match_pr.iterrows():
        if pr_col == 'BOTH':
            pr_amount += float(pr_row.get('Sum EETax', 0) or 0)
            pr_amount += float(pr_row.get('Sum ERTax', 0) or 0)
        elif pr_col and pr_col in pr_row.index:
            pr_amount += float(pr_row.get(pr_col, 0) or 0)

    is_liab = any(kw in recon.lower() for kw in LIAB_KW)
    if is_liab:
        variance  = round(gl_net + pr_amount, 2)
        pr_display= round(-pr_amount, 2)
        note      = 'Liability (credit normal): variance = GL + PR'
    else:
        variance  = round(gl_net - pr_amount, 2)
        pr_display= round(pr_amount, 2)
        note      = ''

    status = '✓ Match' if abs(variance) < TOLERANCE else '⚠ Variance'
    sign   = 'GL + PR' if is_liab else 'GL - PR'
    print(f'         → PR amount ({pr_col}): {pr_amount:,.2f}  |  {sign} = Variance {variance:,.2f}  [{status}]{" (liability)" if is_liab else ""}')
    rows_out.append({'Reconciliation Step': recon, 'GL Code': gc, 'GL Title': title,
                     'GL Net Amount': round(gl_net,2), 'PR Amount': pr_display,
                     'Variance': variance, 'Status': status, 'Notes': note})

# Build final table
recon_df  = pd.DataFrame(rows_out).sort_values('Reconciliation Step').reset_index(drop=True)
total_row = {
    'Reconciliation Step': 'TOTAL', 'GL Code': '', 'GL Title': '',
    'GL Net Amount': recon_df['GL Net Amount'].sum().round(2),
    'PR Amount'    : recon_df['PR Amount'].sum().round(2),
    'Variance'     : recon_df['Variance'].sum().round(2),
    'Status': '✓ Match' if abs(recon_df['Variance'].sum()) < TOLERANCE else '⚠ Variance',
    'Notes': 'Grand total',
}
recon_df = pd.concat([recon_df, pd.DataFrame([total_row])], ignore_index=True)

state['results'] = {
    'recon_df' : recon_df, 'gl_pivot'  : gl_pivot,
    'pr_pivot' : pr_pivot, 'gl_mapped' : gl_mapped, 'pr_mapped': pr_mapped,
}

# ── Final summary ─────────────────────────────────────────────────────────────
data      = recon_df[recon_df['Reconciliation Step'] != 'TOTAL']
matched   = (data['Status'] == '✓ Match').sum()
variances = (data['Status'] != '✓ Match').sum()
total_var = data['Variance'].abs().sum()
is_clean  = total_var < TOLERANCE

print()
print('=' * 65)
print('  RECONCILIATION COMPLETE')
print('=' * 65)
print(f'  Total lines  : {len(data)}')
print(f'  ✓ Matched    : {matched}')
print(f'  ⚠ Variances  : {variances}')
print(f'  Total var    : ${total_var:,.2f}')
print(f'  Result       : {"✅ CLEAN — no variances" if is_clean else "⚠ HAS VARIANCES — review table below"}')
print('=' * 65)

def _style_recon(row):
    if row['Reconciliation Step'] == 'TOTAL':
        return ['background-color:#D9D9D9;font-weight:bold'] * len(row)
    elif '✓ Match' in str(row.get('Status', '')):
        return ['background-color:#C6EFCE'] * len(row)
    else:
        return ['background-color:#FFC7CE'] * len(row)

display(
    recon_df.style
    .apply(_style_recon, axis=1)
    .format({'GL Net Amount': '${:,.2f}', 'PR Amount': '${:,.2f}', 'Variance': '${:,.2f}'}, na_rep='')
    .set_caption('Reconciliation Results  |  🟢 Match   🔴 Variance   ⬜ Total')
    .set_properties(**{'font-size': '11px'})
)

---
## Step 5 — Download Excel Report
Generates a **6-sheet Excel** workbook and downloads it automatically.

In [ ]:
# ── Cell 6: Export & Download Excel ──────────────────────────────────────────
import xlsxwriter

assert state['results'], '❌ No results — run Step 4 first'

res      = state['results']
cfg_rows = state['config_rows']

# ── Colours ───────────────────────────────────────────────────────────────────
C_HDR = '#1F3864'; C_WHT = '#FFFFFF'; C_ALT = '#EEF4FB'
C_M_BG = '#C6EFCE'; C_M_FG = '#276221'
C_V_BG = '#FFC7CE'; C_V_FG = '#9C0006'
C_TOT  = '#D9D9D9'

AMT_FRAGS = ['amount','amt','earn','bene','deduc','tax','net','debit','credit','variance','balance']
def is_amt(c): return any(f in c.lower() for f in AMT_FRAGS)

tag = ''
if CLIENT_NAME and CLIENT_NAME.lower() != 'default': tag += f' | {CLIENT_NAME}'
if PERIOD_LABEL: tag += f' | {PERIOD_LABEL}'

output = io.BytesIO()
wb     = xlsxwriter.Workbook(output, {'in_memory': True})

f_hdr  = wb.add_format({'bold':True,'font_color':C_WHT,'bg_color':C_HDR,'border':1,'align':'center','font_size':10,'valign':'vcenter'})
f_sub  = wb.add_format({'bold':True,'bg_color':'#BDD7EE','border':1,'align':'center','font_size':10})
f_txt  = wb.add_format({'border':1,'font_size':10})
f_alt  = wb.add_format({'border':1,'font_size':10,'bg_color':C_ALT})
f_num  = wb.add_format({'border':1,'num_format':'#,##0.00_);(#,##0.00)','font_size':10})
f_numa = wb.add_format({'border':1,'num_format':'#,##0.00_);(#,##0.00)','bg_color':C_ALT,'font_size':10})
f_match= wb.add_format({'bold':True,'font_color':C_M_FG,'bg_color':C_M_BG,'border':1,'font_size':10})
f_var  = wb.add_format({'bold':True,'font_color':C_V_FG,'bg_color':C_V_BG,'border':1,'font_size':10})
f_tot  = wb.add_format({'bold':True,'bg_color':C_TOT,'border':1,'num_format':'#,##0.00_);(#,##0.00)','font_size':10})
f_tott = wb.add_format({'bold':True,'bg_color':C_TOT,'border':1,'font_size':10})
f_step = wb.add_format({'bold':True,'bg_color':'#D6E4F7','border':1,'font_color':'#1F3864','font_size':10})
f_code = wb.add_format({'bold':True,'bg_color':'#E8F5E9','font_color':'#1B5E20','border':1,'font_size':10})
f_type = wb.add_format({'bold':True,'bg_color':'#FFF3E0','font_color':'#C15700','border':1,'align':'center','font_size':10})

def col_width(series, col_name):
    try:    mx = max(len(str(col_name)), int(series.astype(str).str.len().max()))
    except: mx = len(str(col_name))
    return min(max(mx + 2, 10), 50)

def write_sheet(ws, df, title='', recon=False):
    off = 0
    if title:
        ws.merge_range(0, 0, 0, max(len(df.columns)-1, 0), title, f_sub)
        off = 1
    for ci, col in enumerate(df.columns):
        ws.write(off, ci, col, f_hdr)
    ws.freeze_panes(off + 1, 0)
    for ri, (_, row) in enumerate(df.iterrows()):
        er  = off + 1 + ri
        alt = ri % 2 == 1
        tot = str(row.get('Reconciliation Step', '')).strip().upper() == 'TOTAL'
        for ci, col in enumerate(df.columns):
            val = row[col]
            if tot:
                fmt = f_tot if is_amt(col) else f_tott
            elif recon and 'Status' in df.columns:
                st  = str(row.get('Status', ''))
                fmt = (f_match if '✓ Match' in st else
                       f_var   if ('Variance' in st or 'No PR' in st) else
                       f_numa  if (alt and is_amt(col)) else
                       f_num   if is_amt(col) else
                       f_alt   if alt else f_txt)
            else:
                fmt = (f_numa if (alt and is_amt(col)) else
                       f_num  if is_amt(col) else
                       f_alt  if alt else f_txt)
            if pd.isna(val):                    ws.write_blank(er, ci, None, fmt)
            elif isinstance(val, (int, float)): ws.write_number(er, ci, float(val), fmt)
            else:                               ws.write_string(er, ci, str(val), fmt)
    for ci, col in enumerate(df.columns):
        ws.set_column(ci, ci, col_width(df[col], col))

# ── Sheets 1-5 ────────────────────────────────────────────────────────────────
write_sheet(wb.add_worksheet('GL_Mapped'),      res['gl_mapped'], f'General Ledger — PRS Transactions{tag}')
write_sheet(wb.add_worksheet('PR_Mapped'),      res['pr_mapped'], f'Payroll Register{tag}')
write_sheet(wb.add_worksheet('GL_Pivot'),       res['gl_pivot'],  f'GL Pivot — Net Amount by Reconciliation Step{tag}')
write_sheet(wb.add_worksheet('PR_Pivot'),       res['pr_pivot'],  f'Payroll Register Pivot — Amounts by Code Type{tag}')
write_sheet(wb.add_worksheet('Reconciliation'), res['recon_df'],  f'Payroll Reconciliation Report{tag}', recon=True)

# ── Sheet 6: Payroll Process ──────────────────────────────────────────────────
ws6 = wb.add_worksheet('Payroll_Process')
ws6.merge_range(0, 0, 0, 6, f'Payroll Process — Reconciliation Mapping Configuration{tag}', f_sub)
pp_hdrs = ['Reconciliation Step','GL Code','GL Title','Pay Code','Pay Code Title','Amount Column','Code Type']
pp_wids = [40, 10, 32, 14, 28, 16, 12]
for ci, (h, w) in enumerate(zip(pp_hdrs, pp_wids)):
    ws6.write(1, ci, h, f_hdr)
    ws6.set_column(ci, ci, w)
ws6.freeze_panes(2, 0)

last_step, er, di = None, 2, 0
for row in cfg_rows:
    step = str(row.get('recon_step', '')).strip()
    if step and step != last_step:
        ws6.merge_range(er, 0, er, 6, step, f_step)
        er += 1; last_step = step
    b = f_alt if di % 2 == 1 else f_txt
    ws6.write(er, 0, step,                                b)
    ws6.write(er, 1, str(row.get('gl_code',        '')),  f_code)
    ws6.write(er, 2, str(row.get('gl_title',       '')),  b)
    ws6.write(er, 3, str(row.get('pay_code',       '')),  f_code)
    ws6.write(er, 4, str(row.get('pay_code_title', '')),  b)
    ws6.write(er, 5, str(row.get('amount_column',  '')),  b)
    ws6.write(er, 6, str(row.get('code_type',      '')),  f_type)
    er += 1; di += 1

wb.close()
output.seek(0)

# ── Download ──────────────────────────────────────────────────────────────────
ts      = datetime.datetime.now().strftime('%Y%m%d_%H%M')
safe_cl = re.sub(r'[^\w]', '_', CLIENT_NAME) if CLIENT_NAME.lower() != 'default' else ''
safe_pd = re.sub(r'[^\w]', '_', PERIOD_LABEL) if PERIOD_LABEL else 'report'
outname = f"Payroll_Recon_{''+safe_cl+'_' if safe_cl else ''}{safe_pd}_{ts}.xlsx"

with open(outname, 'wb') as f:
    f.write(output.read())

files.download(outname)
print(f'✅ Downloaded: {outname}')
print(f'   Sheets: GL_Mapped | PR_Mapped | GL_Pivot | PR_Pivot | Reconciliation | Payroll_Process')
